In [1]:
from pathlib import Path
import sys
import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "data_public.csv"
MODELS_DIR = PROJECT_ROOT / "artifacts" / "tuned_models"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH exists:", DATA_PATH.exists())
print("MODELS_DIR exists:", MODELS_DIR.exists())

# 1. Lấy danh sách TẤT CẢ các file model .pkl
model_paths = list(MODELS_DIR.glob("tuned_*.pkl"))
print(f"📌 Tìm thấy {len(model_paths)} mô hình trong thư mục.")
if not model_paths:
    raise FileNotFoundError("Không tìm thấy file model phù hợp")

# --- DỮ LIỆU CẦN DỰ ĐOÁN ---
data_new = pd.DataFrame(
    [
        {
            "Location": "Bình Thạnh, Hồ Chí Minh",
            "Area": 80,
            "Bedrooms": 3,
            "Bathrooms": 3,
            "Floors": 3,
            "Direction": "Đông",
            "Property Type": "Nhà",
            "Road Type": "Đường",
            "Position": "Mặt tiền",
            "Width": 4.5,
            "Length": 18,
            "Alley Width": 0,
            "Agent Listing Count": 5,
        },
        # {
        #     "Area": 55,
        #     "Bedrooms": 2,
        #     "Bathrooms": 2,
        #     "Floors": 2,
        #     "Latitude": 10.838,
        #     "Longitude": 106.665,
        #     "Direction": "Tây",
        #     "Property Type": "Nhà",
        #     "Road Type": "Hẻm",
        #     "Position": "Hẻm",
        #     "Width": 4,
        #     "Length": 14,
        #     "Alley Width": 3,
        #     "Agent Listing Count": 2,
        # },
        # {
        #     "Area": 120,
        #     "Bedrooms": 0,
        #     "Bathrooms": 0,
        #     "Floors": 0,
        #     "Latitude": 10.880,
        #     "Longitude": 106.600,
        #     "Direction": "Nam",
        #     "Property Type": "Đất",
        #     "Road Type": "Đường",
        #     "Position": "Mặt tiền",
        #     "Width": 6,
        #     "Length": 20,
        #     "Alley Width": 0,
        #     "Agent Listing Count": 1,
        # },
    ]
)

# Tạo một DataFrame riêng để lưu kết quả so sánh, lấy dữ liệu gốc làm nền
comparison_df = data_new.copy()

# --- VÒNG LẶP CHẠY TỪNG MÔ HÌNH ---
for path in model_paths:
    # Lấy tên mô hình từ tên file (ví dụ: tuned_random_forest.pkl -> random_forest)
    model_name = path.stem.replace("tuned_", "").upper()
    print(f"\n🔄 Đang xử lý mô hình: {model_name}...")

    try:
        # 2. Load bundle
        bundle = joblib.load(path)

        # 3. Bung các thành phần
        model = bundle["model"]
        encode_pipeline = bundle["encode_pipeline"]
        preprocessor = bundle["preprocessor"]
        log_target = bundle["log_target"]

        # 4. Transform dữ liệu (Làm sạch & chuẩn hóa)
        X_new = preprocessor.transform(data_new)

        # 4b. Encode + scale theo pipeline
        X_new = encode_pipeline.transform(X_new)

        # 5. Predict
        preds = model.predict(X_new)

        # 6. Đảo ngược log nếu lúc train có dùng log transform biến target
        if log_target:
            final_preds = np.expm1(preds)
        else:
            final_preds = preds

        # Lưu kết quả dự đoán của mô hình này vào bảng so sánh
        comparison_df[f"Predict_{model_name}"] = final_preds
        print(f"✅ Dự đoán xong cho {model_name}")

    except Exception as e:
        print(f"❌ Lỗi khi chạy mô hình {model_name}: {e}")

# --- 7. IN KẾT QUẢ SO SÁNH ---
print("\n" + "=" * 50)
print("📊 BẢNG SO SÁNH KẾT QUẢ DỰ ĐOÁN GIỮA CÁC MÔ HÌNH")
print("=" * 50)

# Định dạng in số thực cho đẹp mắt
pd.set_option("display.float_format", lambda x: "%.2f" % x)

# Chỉ hiển thị một vài thông tin gốc chính kèm với cột kết quả dự đoán của các model
show_cols = ["Area", "Property Type", "Position"] + [
    c for c in comparison_df.columns if c.startswith("Predict_")
]
display(comparison_df[show_cols])

PROJECT_ROOT: C:\Users\ADMIN\OneDrive\Desktop\Housing-Market-Prediction\Nhom20
DATA_PATH exists: True
MODELS_DIR exists: True
📌 Tìm thấy 4 mô hình trong thư mục.

🔄 Đang xử lý mô hình: GRADIENT_BOOSTING...
✅ Dự đoán xong cho GRADIENT_BOOSTING

🔄 Đang xử lý mô hình: LIGHTGBM...


c:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


✅ Dự đoán xong cho LIGHTGBM

🔄 Đang xử lý mô hình: RANDOM_FOREST...
✅ Dự đoán xong cho RANDOM_FOREST

🔄 Đang xử lý mô hình: XGBOOST...
✅ Dự đoán xong cho XGBOOST

📊 BẢNG SO SÁNH KẾT QUẢ DỰ ĐOÁN GIỮA CÁC MÔ HÌNH


,Area,Property Type,Position,Predict_GRADIENT_BOOSTING,Predict_LIGHTGBM,Predict_RANDOM_FOREST,Predict_XGBOOST
0,80,Nhà,Mặt tiền,4350.30,5143.04,5685.67,3783.18
